# Lab 01 – Automated Build & Release Pipeline

**Scenario:** The Week 8 PoC must now deploy through a fully automated CI/CD pipeline. You will scaffold build, test, scan, and promotion stages that package code, prompts, guardrails, and retrieval assets. Deliverables feed the Week 9 CAB review.

## Objectives
- Validate prerequisite tooling (GitHub Actions/ArgoCD/registry access)
- Author a reusable GitHub Actions workflow or equivalent CI pipeline
- Produce signed artifacts (container + prompt bundle) with release metadata
- Simulate promotion to staging and prepare evidence bundles for CAB

In [ ]:
import json
from pathlib import Path

required_dirs = [
    Path('.github/workflows'),
    Path('infra/'),
    Path('pipelines/'),
]
missing = [str(p) for p in required_dirs if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing prerequisite directories: {missing}')

print('Environment validation complete. Ready to scaffold pipeline.')

## Stage 1 – Define Pipeline Blueprint
Update the architecture diagram with the pipeline stages and owners. Capture the mapping in the Week 9 readiness checklist.

In [ ]:
from textwrap import dedent

workflow = dedent('''
name: genai-release

on:
  push:
    branches: [ main ]

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install CI dependencies
        run: pip install -r requirements-ci.txt
      - name: Unit & integration tests
        run: pytest --maxfail=1 --disable-warnings
      - name: Security scan
        run: trivy fs --exit-code 1 .
      - name: Build container image
        run: docker build -t ${{ secrets.REGISTRY }}/assistant:${{ github.sha }} .
      - name: Push image
        run: docker push ${{ secrets.REGISTRY }}/assistant:${{ github.sha }}
      - name: Sign artifact
        run: cosign sign ${{ secrets.REGISTRY }}/assistant:${{ github.sha }}
      - name: Publish release manifest
        run: python scripts/publish_release_manifest.py --sha ${{ github.sha }}
''')

output_path = Path('.github/workflows/genai-release.yaml')
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(workflow)
print(f'Scaffolded workflow at {output_path}')

## Stage 2 – Promotion Simulation
Populate the staging ArgoCD manifest with the newly generated image tag and release metadata. Record the simulated approval in the CAB evidence log.

In [ ]:
from datetime import datetime

release_id = datetime.utcnow().strftime('rel-%Y%m%d-%H%M')
manifest_path = Path('env/staging/values.yaml')
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(f'imageTag: {release_id}
releaseSha: PLACEHOLDER_SHA
')

evidence = {
    'release_id': release_id,
    'artifact_image': '${REGISTRY}/assistant:PLACEHOLDER_SHA',
    'approver': 'automation@cab',
    'timestamp_utc': datetime.utcnow().isoformat(),
}
Path('artifacts').mkdir(exist_ok=True)
Path('artifacts/release-evidence.json').write_text(json.dumps(evidence, indent=2))
print('Promotion manifest and evidence prepared.')

## Stage 3 – Regression & Security Evidence
Run the integrated test suite, red-team regression, and security scans. Attach the Langfuse trace bundle and Trivy report to the artifacts folder.

In [ ]:
def run_placeholder(command: str) -> None:
    print(f'[stub] Execute: {command}')

run_placeholder('pytest tests/')
run_placeholder('redteam.harness run --config guardrails/prompt-firewall.yaml')
run_placeholder('trivy image ${REGISTRY}/assistant:PLACEHOLDER_SHA --format json --output artifacts/trivy.json')
print('Record actual outputs when executing in CI environment.')

## Stage 4 – Readiness Handoff
Update the following artifacts before submitting the lab:
- `resources/deployment-readiness-checklist.md` (Week 9 section)
- `resources/production-slo-scorecard.csv` (Baseline columns)
- CAB ticket with release ID and evidence links
- Screenshots of CI pipeline run + artifact registry entries

## Retrospective
Complete the team reflection. Capture blockers, automation backlog items, and next steps for production readiness.

In [ ]:
from tabulate import tabulate

rows = [
    ['What worked', ''],
    ['Pipeline gaps', ''],
    ['Security findings', ''],
    ['Next actions', ''],
]
table = tabulate(rows, headers=['Topic', 'Notes'], tablefmt='github')
print(table)
print('Fill in the retrospective table and commit to the repo.')